## Fram Strait 2025

This handler converts the [FS2025](https://zenodo.org/records/20761832) I-129 seawater CSV record from Zenodo into MARIS-standard NetCDF4. The source table contains station, position, collection date, hydrographic metadata, a provider sample ID, and I-129 values with absolute uncertainties.

The handler is structured to accept further compatible Fram Strait CSV records through `RECORDS`; each input is read and combined into the `SEAWATER` group.

In [ ]:
#| default_exp handlers.fram_strait2025

In [ ]:
#| export
from fastcore.all import *
import pandas as pd
import numpy as np
import requests
import io
import gsw

from marisco.callbacks import PerGroupCB, Transformer, EncodeTimeCB, SanitizeLonLatCB, RemapCB, AddSampleIDCB
from marisco.metadata import GlobAttrsFeeder, BboxCB, DepthRangeCB, TimeRangeCB, KeyValuePairCB
from marisco.encoders import NetCDFEncoder
from marisco.nc2csv import to_csv
from marisco.callbacks import get_lut

In [ ]:
#| exports
RECORDS = {
    "FS2025_i129": {
        "url": "https://zenodo.org/records/20761832/files/FS2025_i129.csv?download=1",
    },
}

fname_out = "Fram_Strait_2025.nc"
src_dir = None

In [ ]:
#| export
def load_data(
    recs=None,  # Optional record mapping; defaults to all RECORDS
) -> dict:
    "Fetch Fram Strait CSV records and return one combined SEAWATER DataFrame"
    recs = recs or RECORDS
    parts = []

    for record in recs.values():
        resp = requests.get(record["url"], timeout=60)
        resp.raise_for_status()

        df = pd.read_csv(
            io.BytesIO(resp.content),
            encoding="utf-8-sig",
        )
        parts.append(df)

    return {"SEAWATER": pd.concat(parts, ignore_index=True)}

In [ ]:
#| eval: false
dfs = load_data()

In [ ]:
#|eval: false
dfs['SEAWATER'].columns

Index(['Cruise', 'Station', 'Latitude_degN', 'Longitude_degE', 'Date',
       'Niskin', 'Pressure_dbar', 'PracticalSalinity', 'Temperature_degC',
       'Sample_ID', 'I129_at_l', 'unc_I129_at_l'],
      dtype='str')

In [ ]:
#|eval: false
print(dfs['SEAWATER'].describe(include='number').T[['count', 'mean', 'min', 'max']])

                   count          mean           min           max
Station            177.0  3.723107e+02  3.410000e+02  4.150000e+02
Latitude_degN      177.0  7.895424e+01  7.883217e+01  8.040850e+01
Longitude_degE     177.0 -4.121274e+00 -1.340367e+01  8.000000e+00
Niskin             177.0  1.021469e+01  1.000000e+00  2.300000e+01
Pressure_dbar      177.0  2.675278e+02  4.524000e+00  2.621694e+03
PracticalSalinity  177.0  3.379420e+01  2.898850e+01  3.510550e+01
Temperature_degC   177.0  8.848593e-01 -1.815000e+00  9.014500e+00
Sample_ID          177.0  9.009040e+01  1.000000e+00  1.800000e+02
I129_at_l          177.0  3.038391e+09  1.502199e+08  7.150855e+09
unc_I129_at_l      177.0  7.779234e+07  4.745141e+06  1.819380e+08


## Column renaming, time parsing, and depth conversion

FS2025 is seawater-only. `RenameColsCB` maps provider metadata columns to MARIS working names, `ParseDateTimeCB` converts the collection date to UTC `TIME`, and `AddDepthCB` computes sampling depth from the reported `Pressure_dbar` using the TEOS-10 equation of state via the `gsw` library.

In [ ]:
#| export
class RenameColsCB(PerGroupCB):
    "Map FS2025 provider columns to MARIS standard names"
    def each_grp(self, grp, df, tfm):
        tfm.dfs[grp] = df.rename(columns={
            "Station": "STATION",
            "Latitude_degN": "LAT",
            "Longitude_degE": "LON",
            "PracticalSalinity": "SAL",
            "Temperature_degC": "TEMP",
            "Sample_ID": "SMP_ID_PROVIDER",
        })

In [ ]:
# Verify RenameColsCB maps provider columns to MARIS names
dfs_mock = {
    "SEAWATER": pd.DataFrame({
        "Station": [341],
        "Latitude_degN": [78.832167],
        "Longitude_degE": [-2.004667],
        "PracticalSalinity": [34.9],
        "Temperature_degC": [0.0538],
        "Sample_ID": [1],
    })
}

tfm = Transformer(dfs_mock, cbs=[RenameColsCB()])
tfm()

for col in ["STATION", "LAT", "LON", "SAL", "TEMP", "SMP_ID_PROVIDER"]:
    test_eq(col in tfm.dfs["SEAWATER"].columns, True)

print("RenameColsCB: FS2025 columns mapped correctly. ✓")

RenameColsCB: FS2025 columns mapped correctly. ✓


In [ ]:
#| eval: false
tfm = Transformer(dfs, cbs=[RenameColsCB()])
tfm()
print(
    tfm.dfs["SEAWATER"][
        ["LAT", "LON", "STATION", "SAL", "TEMP", "SMP_ID_PROVIDER"]
    ].head(2).to_string()
)

         LAT       LON  STATION     SAL    TEMP  SMP_ID_PROVIDER
0  78.832167 -2.004667      341  34.900  0.0538                1
1  78.832167 -2.004667      341  34.917  0.8933                2


In [ ]:
#| export
class ParseDateTimeCB(PerGroupCB):
    "Parse FS2025 collection date into a UTC TIME value"
    def each_grp(self, grp, df, tfm):
        tfm.dfs[grp] = df.assign(
            TIME=pd.to_datetime(df["Date"], format="%Y-%m-%d", utc=True)
        ).drop(columns="Date")

In [ ]:
# Verify ParseDateTimeCB parses Date into UTC TIME
dfs_mock = {"SEAWATER": pd.DataFrame({"Date": ["2025-07-30"]})}

tfm = Transformer(dfs_mock, cbs=[ParseDateTimeCB()])
tfm()

test_eq("TIME" in tfm.dfs["SEAWATER"].columns, True)
test_eq("Date" not in tfm.dfs["SEAWATER"].columns, True)
print(f"ParseDateTimeCB: TIME = {tfm.dfs['SEAWATER']['TIME'].iloc[0]}. ✓")

ParseDateTimeCB: TIME = 2025-07-30 00:00:00+00:00. ✓


In [ ]:
#| eval: false
tfm = Transformer(dfs, cbs=[
    RenameColsCB(),
    ParseDateTimeCB()
])
tfm()

print(tfm.dfs["SEAWATER"][["TIME"]].head(3).to_string())

                       TIME
0 2025-07-30 00:00:00+00:00
1 2025-07-30 00:00:00+00:00
2 2025-07-30 00:00:00+00:00


In [ ]:
tfm.dfs["SEAWATER"].head()

,Cruise,STATION,LAT,LON,Niskin,Pressure_dbar,SAL,TEMP,SMP_ID_PROVIDER,I129_at_l,unc_I129_at_l,TIME
0,FS2025,341,78.832167,-2.004667,3,1000.560,34.900,0.0538,1,2.264769e+09,5.886864e+07,2025-07-30 00:00:00+00:00
1,FS2025,341,78.832167,-2.004667,4,750.210,34.917,0.8933,2,1.951592e+09,5.074921e+07,2025-07-30 00:00:00+00:00
2,FS2025,341,78.832167,-2.004667,5,500.641,34.923,1.6611,3,2.081041e+09,5.406397e+07,2025-07-30 00:00:00+00:00
3,FS2025,341,78.832167,-2.004667,6,400.526,34.948,2.2196,4,2.238436e+09,5.812364e+07,2025-07-30 00:00:00+00:00
4,FS2025,341,78.832167,-2.004667,8,250.385,34.987,3.0336,5,2.355675e+09,6.119206e+07,2025-07-30 00:00:00+00:00


:::{.callout-important}
## FEEDBACK TO DATA PROVIDER

We use the [Thermodynamic Equation Of Seawater - 2010 (TEOS-10)](https://www.teos-10.org) via the `gsw` Python package (`gsw.z_from_p`) to convert your reported `Pressure_dbar` to sampling depth in metres. The conversion uses the reported latitude for the gravitational acceleration correction. Do you confirm this is the correct approach and conversion for these samples? We have rounded depths to one decimal place — please let us know if your convention uses a different precision.
:::

In [ ]:
#| export
class AddDepthCB(PerGroupCB):
    "Compute sampling depth using Thermodynamic Equation of SeaWater 2010 (TEOS-10)"
    def each_grp(self, grp, df, tfm): 
        df["SMP_DEPTH"] = np.round(-gsw.z_from_p(df['Pressure_dbar'], df['LAT']), 1)

In [ ]:
#| eval: false
tfm = Transformer(dfs, cbs=[
    RenameColsCB(),
    ParseDateTimeCB(),
    AddDepthCB()
])
tfm()

print(tfm.dfs["SEAWATER"].SMP_DEPTH)

0      987.6
1      741.0
2      494.8
3      395.9
4      247.6
       ...  
172    198.3
173    148.6
174     99.1
175     49.6
176      4.9
Name: SMP_DEPTH, Length: 177, dtype: float64


## Nuclide measurements

FS2025 reports a single radionuclide, I-129, in atoms per litre. The MARIS lookup tables map `i129` to NUCLIDE ID 28 and `atom per l` to UNIT ID 12:

In [ ]:
#| eval: false
nuclides = get_lut('NUCLIDE')
nuclides['i129']

28

In [ ]:
#| eval: false
units = get_lut('UNIT')
units['atom per l']

12

`VALUE` and `UNC` copy the provider columns directly; no conversion is needed. Four callbacks assign each column:

In [ ]:
#| export
class AddNuclideCB(PerGroupCB):
    "Assign NUCLIDE MARIS ID for I-129"
    def each_grp(self, grp, df, tfm): 
        tfm.dfs[grp] = df.assign(NUCLIDE=28)

In [ ]:
#| export
class AddValueCB(PerGroupCB):
    "Assign VALUE column from the I-129 atoms per litre measurement"
    def each_grp(self, grp, df, tfm):
        tfm.dfs[grp] = df.assign(VALUE=df['I129_at_l'])

In [ ]:
#| export
class AddUnitCB(PerGroupCB):
    "Assign UNIT MARIS ID for atoms per litre"
    def each_grp(self, grp, df, tfm): 
        tfm.dfs[grp] = df.assign(UNIT=12)

In [ ]:
#| export
class AddUncertCB(PerGroupCB):
    "Assign UNC column from the I-129 provider uncertainty"
    def each_grp(self, grp, df, tfm):
        tfm.dfs[grp] = df.assign(UNC=df['unc_I129_at_l'])

The provider does not report a detection level, but MARIS requires this field. The available MARIS categories are:

In [ ]:
#| eval: false
get_lut('DL')

{'Not applicable': -1,
 'Not available': 0,
 'Detected value': 1,
 'Detection limit': 2,
 'Not detected': 3,
 'Derived': 4}

In [ ]:
#| export
class AddDetectionLimitCB(PerGroupCB):
    "Assign missing Detection Limit column to MARIS 'Detected value: 1' category"
    def each_grp(self, grp, df, tfm): 
        tfm.dfs[grp] = df.assign(DL=1)

## Analysis laboratory

The MARIS LAB LUT maps the analysing laboratory to ID 345:

In [ ]:
#| eval: false
labs = get_lut('LAB')
labs['Laboratory of Ion Beam Physics _LIP_, ETZ Zürich, Switzerland']

345

In [ ]:
#| export
class AddLabCB(PerGroupCB):
    "Assign LAB MARIS ID for LIP, ETH Zürich"
    def each_grp(self, grp, df, tfm):
        tfm.dfs[grp] = df.assign(LAB=345)

Cumulative pipeline through the new column callbacks:

In [ ]:
#| eval: false
tfm = Transformer(dfs, cbs=[
    RenameColsCB(),
    ParseDateTimeCB(),
    AddDepthCB(),
    AddNuclideCB(),
    AddValueCB(),
    AddUnitCB(),
    AddUncertCB(),
    AddDetectionLimitCB(),
    AddLabCB(),
])
tfm()

tfm.dfs["SEAWATER"][["NUCLIDE", "UNIT", "VALUE", "UNC", "LAB", "DL"]].head()

,NUCLIDE,UNIT,VALUE,UNC,LAB,DL
0,28,12,2.264769e+09,5.886864e+07,345,1
1,28,12,1.951592e+09,5.074921e+07,345,1
2,28,12,2.081041e+09,5.406397e+07,345,1
3,28,12,2.238436e+09,5.812364e+07,345,1
4,28,12,2.355675e+09,6.119206e+07,345,1


## Standardise final columns


- `SanitizeLonLatCB`: validates lat/lon ranges and corrects sign convention
- `EncodeTimeCB`: encodes `TIME` into the NetCDF numeric representation
- `AddSampleIDCB`: assigns sequential `SMP_ID`, preserves `SMP_ID_PROVIDER`

All three are imported from `marisco.callbacks` and need no FS2025-specific configuration.

**Cast STATION to string before encoding**

FS2025's `Station` column is pure numeric, so pandas infers `int64`. But `STATION` maps to a `string`-typed NetCDF variable, so `FormatStationCB` casts it to `str` as the last step before encoding. This callback is defined locally in this notebook only.

In [ ]:
#| export
class FormatStationCB(PerGroupCB):
    "Cast STATION to str for the NetCDF4 string-typed station variable"
    def each_grp(self, grp, df, tfm):
        df["STATION"] = df["STATION"].astype(str)

In [ ]:
# Verify FormatStationCB casts STATION to str
dfs_mock = {'SEAWATER': pd.DataFrame({'STATION': [341, 415]})}
tfm = Transformer(dfs_mock, cbs=[FormatStationCB()])
tfm()
out = tfm.dfs['SEAWATER']
test_eq(out['STATION'].tolist(), ['341', '415'])
test_eq(all(isinstance(v, str) for v in out['STATION']), True)  # what the encoder's per-element NetCDF write actually needs
print("FormatStationCB: STATION cast to str. ✓")

FormatStationCB: STATION cast to str. ✓


In [ ]:
#| eval: false
tfm = Transformer(dfs, cbs=[
    RenameColsCB(),
    ParseDateTimeCB(),
    AddDepthCB(),
    AddNuclideCB(),
    AddValueCB(),
    AddUnitCB(),
    AddUncertCB(),
    AddDetectionLimitCB(),
    AddLabCB(),
    SanitizeLonLatCB(),
    EncodeTimeCB(),
    AddSampleIDCB(col_provider="SMP_ID_PROVIDER"),
    FormatStationCB(),
])
tfm()
out = tfm.dfs['SEAWATER']
print(f"Final shape: {out.shape}")
print("Columns:", out.columns.tolist())
print(out[['SMP_ID', 'SMP_ID_PROVIDER', 'NUCLIDE', 'UNIT', 'LAB']].head(4).to_string())

Final shape: (177, 20)


Columns: ['Cruise', 'STATION', 'LAT', 'LON', 'Niskin', 'Pressure_dbar', 'SAL', 'TEMP', 'SMP_ID_PROVIDER', 'I129_at_l', 'unc_I129_at_l', 'TIME', 'SMP_DEPTH', 'NUCLIDE', 'VALUE', 'UNIT', 'UNC', 'DL', 'LAB', 'SMP_ID']


   SMP_ID SMP_ID_PROVIDER  NUCLIDE  UNIT  LAB
0       1               1       28    12  345
1       2               2       28    12  345
2       3               3       28    12  345
3       4               4       28    12  345


In [ ]:
#| eval: false
print("Final data summary (uppercase columns only):")
upper_cols = [c for c in out.columns if c.isupper()]
print(out[upper_cols].describe().to_string())

Final data summary (uppercase columns only):


              LAT         LON         SAL        TEMP          TIME    SMP_DEPTH  NUCLIDE         VALUE   UNIT           UNC     DL    LAB      SMP_ID
count  177.000000  177.000000  177.000000  177.000000  1.770000e+02   177.000000    177.0  1.770000e+02  177.0  1.770000e+02  177.0  177.0  177.000000
mean    78.954240   -4.121274   33.794202    0.884859  1.754350e+09   264.209605     28.0  3.038391e+09   12.0  7.779234e+07    1.0  345.0   89.000000
std      0.412695    5.923451    1.639711    2.362646  3.917809e+05   366.434204      0.0  1.330420e+09    0.0  3.403782e+07    0.0    0.0   51.239633
min     78.832167  -13.403667   28.988500   -1.815000  1.753834e+09     4.500000     28.0  1.502199e+08   12.0  4.745141e+06    1.0  345.0    1.000000
25%     78.833000   -8.998833   33.023500   -0.956300  1.754006e+09    49.500000     28.0  2.114304e+09   12.0  5.412238e+07    1.0  345.0   45.000000
50%     78.833333   -4.002000   34.699000    0.321000  1.754179e+09   148.800000     28.0  2.6

## NetCDF encoder

The encoder wraps the full pipeline and writes the standardised data to a NetCDF4 file. Global attributes are assembled via `GlobAttrsFeeder` with `BboxCB`, `DepthRangeCB`, `TimeRangeCB`, plus keywords and processing logs.

The resulting file contains spatial, depth, and time coverage derived from the transformed seawater data, together with FS2025 keywords and the recorded processing steps.

In [ ]:
#| exports
FS2025_KEYWORDS = [
    "Fram Strait","Greenland Sea","I-129","radionuclides","seawater","Arctic Ocean",
]

def get_attrs(tfm):
    "Retrieve global attributes for Fram Strait 2025"
    return GlobAttrsFeeder(tfm.dfs, cbs=[
        BboxCB(),
        DepthRangeCB(),
        TimeRangeCB(),
        KeyValuePairCB("keywords", ", ".join(FS2025_KEYWORDS)),
        KeyValuePairCB("publisher_postprocess_logs", ", ".join(tfm.logs)),
    ])()

In [ ]:
#| exports
def encode(fname_out=None  # Output NetCDF file path; defaults to fname_out
            ):
    "Encode Fram Strait 2025 data to NetCDF4"
    fname_out = fname_out or globals().get("fname_out", "Fram_Strait_2025.nc")
    dfs = load_data()
    tfm = Transformer(dfs, cbs=[
        RenameColsCB(),
        ParseDateTimeCB(),
        AddDepthCB(),
        AddNuclideCB(),
        AddValueCB(),
        AddUnitCB(),
        AddUncertCB(),
        AddDetectionLimitCB(),
        AddLabCB(),
        SanitizeLonLatCB(),
        EncodeTimeCB(),
        AddSampleIDCB(col_provider="SMP_ID_PROVIDER"),
        FormatStationCB()
        ])
    tfm()
    encoder = NetCDFEncoder(tfm.dfs, dest_fname=fname_out,
                            global_attrs=get_attrs(tfm))
    encoder.encode()

In [ ]:
#| eval: false
# Encode to NetCDF
encode("../../_data/output/fram_strait2025.nc")
print("Fram Strait 2025 NetCDF written.")

Fram Strait 2025 NetCDF written.


In [ ]:
#| eval: false
to_csv("../../_data/output/fram_strait2025.nc")

[Path('../../_data/output/fram_strait2025_SEAWATER.csv')]

In [ ]:
#| eval: false
df = pd.read_csv("../../_data/output/fram_strait2025_SEAWATER.csv")
print(f"Shape: {df.shape}")
print(f"Columns: {df.columns.tolist()}")
print(f"\nNuclide IDs: {df.nuclide_id.unique()}")
print(f"Unit IDs: {df.unit_id.unique()}")
print(f"Sample type IDs: {df.samptype_id.unique()}")
print(f"\nStation range: {df.station.min()}–{df.station.max()}")
print(f"Date range: {df.begperiod.min()} to {df.begperiod.max()}")
print(f"Sample depth range: {df.sampdepth.min()} to {df.sampdepth.max()}m")

Shape: (177, 15)


Columns: ['detection', 'lab_id', 'latitude', 'longitude', 'nuclide_id', 'salinity', 'sampdepth', 'samplabcode', 'station', 'temperatur', 'begperiod', 'uncertaint', 'unit_id', 'activity', 'samptype_id']



Nuclide IDs: [28]


Unit IDs: [12]


Sample type IDs: [1]



Station range: 341–415


Date range: 2025-07-30 to 2025-08-13


Sample depth range: 4.5 to 2578.0m


In [ ]:
#| eval: false
print(df.head())

  detection  lab_id  latitude  longitude  nuclide_id  salinity  sampdepth  \
0         =     345  78.83217  -2.004667          28    34.900      987.6   
1         =     345  78.83217  -2.004667          28    34.917      741.0   
2         =     345  78.83217  -2.004667          28    34.923      494.8   
3         =     345  78.83217  -2.004667          28    34.948      395.9   
4         =     345  78.83217  -2.004667          28    34.987      247.6   

   samplabcode  station  temperatur   begperiod  uncertaint  unit_id  \
0            1      341      0.0538  2025-07-30  58868640.0       12   
1            2      341      0.8933  2025-07-30  50749210.0       12   
2            3      341      1.6611  2025-07-30  54063970.0       12   
3            4      341      2.2196  2025-07-30  58123640.0       12   
4            5      341      3.0336  2025-07-30  61192064.0       12   

       activity  samptype_id  
0  2.264768e+09            1  
1  1.951592e+09            1  
2  2.081041